# From Hugging Face Hub to robot hardware

Notebook companion to the blog post of the same name. Walks through the LeRobot integration in **Strands Robots** end to end:

1. Build a Strands agent (Claude Opus 4.8 on Bedrock by default) with the LeRobot AgentTools and a `Robot('so100')` simulation.
2. Record a demonstration as a `LeRobotDataset` (local cache, optional Hub push).
3. Run a policy on the same robot, ask the agent to render the final frame.
4. (Reference only) Deploy the same agent code to a physical SO-101 by flipping `mode='real'`.
5. Broadcast a command to every peer on the local Zenoh mesh.

The defaults run fully in simulation with the Mock policy: no GPU, no Docker, no Hub credentials required. To use a trained policy or push to your Hub account, see the notes in the relevant cells.

**Repository:** https://github.com/strands-labs/robots  
**Blog post:** *From Hugging Face Hub to robot hardware with Strands Agents and LeRobot*

## Prerequisites

```bash
pip install "strands-robots[sim-mujoco,lerobot,mesh]" strands-agents
```

Python 3.12+. The first run downloads MuJoCo Menagerie assets for the SO-100 (~30 seconds, one-time).

Environment variables (dev/lab posture for the local mesh):

```bash
export STRANDS_MESH_LOCAL_DEV=1
```

AWS Bedrock access for Claude Opus 4.8 (`global.anthropic.claude-opus-4-8`). If your access is in a different region or account, override `MODEL_ID` and `AWS_REGION` in the next cell.

In [ ]:
import logging
import os
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(name)s] %(levelname)s: %(message)s", datefmt="%H:%M:%S")

# Dev/lab mesh posture so Step 5 has a mesh to talk to.
os.environ.setdefault("STRANDS_MESH_LOCAL_DEV", "1")

# Bedrock model + region. Verify the exact ID in your AWS Bedrock console.
# AWS_REGION resolves from your AWS environment; set it explicitly here
# (e.g. "us-east-1") if you need to override.
MODEL_ID = "global.anthropic.claude-opus-4-8"
AWS_REGION = os.environ.get("AWS_REGION") or os.environ.get("AWS_DEFAULT_REGION")

# Recording target. Set HF_USER to push the dataset to your Hub account.
HF_USER = None  # e.g. "my_username"
DATASET_NAME = "strands-cube-pick"
REPO_ID = f"{HF_USER}/{DATASET_NAME}" if HF_USER else f"local/{DATASET_NAME}"
PUSH_TO_HUB = HF_USER is not None
TASK = "pick up the red cube"

## Step 1: Build the agent

One `Robot('so100')` call returns a MuJoCo simulation by default. Pass `mode='real'` to get a hardware-backed robot driven by LeRobot, and the rest of the workflow is identical.

The agent gets two tools: the `Robot` itself (handles simulation or hardware uniformly) and `robot_mesh` (peer-to-peer fleet messaging). LeRobot's CLIs handle calibration and hardware teleop outside this notebook.

In [ ]:
from strands import Agent
from strands.models import BedrockModel

from strands_robots import Robot, robot_mesh

model = BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION) if AWS_REGION else BedrockModel(model_id=MODEL_ID)
robot = Robot("so100", data_config="so100_dualcam")  # sim by default

agent = Agent(
    model=model,
    tools=[robot, robot_mesh],
)

## Step 2: Record a demonstration

One agent prompt → one tool sequence → one `LeRobotDataset` episode. The agent composes the scene (red cube + front camera), starts recording, runs the Mock policy, and finalizes the episode with `stop_recording`.

The `Robot()` factory already initialised the simulation world, so the prompt acknowledges that to avoid the agent re-creating it.

In [ ]:
NUM_STEPS = 1000  # ≈ 33s of data at 30fps

push_clause = (
    f"Push the result to {REPO_ID} when done." if PUSH_TO_HUB else "Keep the dataset local; do not push to the Hub."
)

prompt = (
    f"A simulation world is already set up with the so100 robot "
    f"(initialised by the Robot() factory). Compose the scene and "
    f"record one demonstration:\n"
    f"\n"
    f"Scene:\n"
    f"  - Add a small red cube near the robot (about 3cm on a side).\n"
    f"  - Add a front-facing camera named 'front' looking at the cube.\n"
    f"\n"
    f"Recording:\n"
    f"  - Start recording to repo_id={REPO_ID!r} at FPS 30 with task={TASK!r}.\n"
    f"  - Run the Mock policy for {NUM_STEPS} steps with instruction={TASK!r}.\n"
    f"  - Call stop_recording to finalize the episode. {push_clause}"
)

agent(prompt)

### Verify the recording

The dataset is on disk at `~/.cache/huggingface/lerobot/<repo_id>/`. Loading it through the `LeRobotDataset` class confirms the episode and frame counts:

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds = LeRobotDataset(REPO_ID)
cache_path = Path.home() / ".cache" / "huggingface" / "lerobot" / REPO_ID

print(f"Dataset: {REPO_ID}")
print(f"  Location:  {cache_path}")
print(f"  Episodes:  {ds.num_episodes}")
print(f"  Frames:    {ds.num_frames}")
print(f"  FPS:       {ds.fps}")
print(f"  Features:  {list(ds.features.keys())}")

Inside the cache directory you'll find:

- `data/chunk-000/file-000.parquet` holds joint states and actions
- `videos/observation.images.front/chunk-000/file-000.mp4` is the recording
- `meta/` holds schema and episode metadata

Open the MP4 in QuickTime/VLC to see the recorded rollout.

## Step 3: Run a policy on the robot

The same agent now drives the policy. With the Mock policy (the default), this is purely a workflow sanity-check: the arm moves through placeholder actions rather than executing a trained grasp.

For real cube-picking behaviour, switch the prompt to drive `gr00t_inference` (NVIDIA GR00T container, requires Docker + GPU) or `create_policy('lerobot_local', ...)` with a checkpoint trained for the task. See this folder's README for the exact wiring.

In [ ]:
agent(f"Run the Mock policy on the robot for 200 steps with the instruction '{TASK}'. Render the final frame.")

## Step 4: Deploy to physical hardware (reference)

Switching from simulation to a physical SO-101 is one flag at construction time:

```python
robot = Robot(
    "so100",
    mode="real",
    port="/dev/ttyACM0",
    data_config="so100_dualcam",
    cameras={
        "front": {"type": "opencv", "index_or_path": "/dev/video0", "fps": 30},
        "wrist": {"type": "opencv", "index_or_path": "/dev/video2", "fps": 30},
    },
)
```

The rest of the code (`Agent(tools=[robot, ...])`, the recording prompt, the policy prompt) is unchanged. The same `LeRobotDataset` writer handles real-camera frames and hardware joint reads through LeRobot's driver layer.

(Skipped in this notebook because hardware isn't typically attached to a Jupyter kernel.)

## Step 5: Mesh broadcast

Every `Robot()` auto-joins a local Zenoh mesh, so fleets coordinate out of the box. The agent uses the `robot_mesh` tool to enumerate peers and send commands.

In this single-process demo, only the local `so100_sim` joined the mesh, so peer count will be small (likely zero remote peers). On a multi-machine setup, the same prompt would discover and address every robot on the network.

In [ ]:
agent(
    "Use robot_mesh to list every peer and local robot, then broadcast "
    "the instruction 'go to home pose' to each one with a 5-second timeout."
)

## Cleanup

Nothing long-running was started in this Mock-policy run. If you switched the policy to GR00T (which launches a Docker container) or LerobotLocal (which keeps GPU memory resident), the cleanup prompt for those is:

```python
agent("Stop the GR00T inference service on port 5555.")
```

or the equivalent shutdown call on the policy you started.

## Next steps

- **Push to the Hub.** Set `HF_USER` in the second cell, export `HF_TOKEN=hf_...` with write scope, and re-run from Step 2.
- **Train on the dataset.** The `LeRobotDataset` you just produced is in the same on-disk format the upstream LeRobot training CLI expects; `lerobot-train` consumes it directly.
- **Use a real trained policy.** Swap the Mock prompt in Step 3 for a `lerobot_local` or `groot` checkpoint. See this folder's `README.md`.
- **Run on physical hardware.** Calibrate your SO-101 once on the shell with `lerobot-calibrate --robot.type=so101_follower --robot.id=my_follower`, then re-run Step 1 with `mode='real'`.